# Diviser pour régner — Exercice 7 : tourner une image d'un quart de tour

> **Énoncé (résumé)**
> L'image `turnbuffon.png` (512 × 512 pixels) a été tournée d'un quart de tour. Le module `turnbuffon.py` (fourni par le professeur, il utilise *pygame*) donne :
> `get_pixel(i, j)`, `set_pixel(i, j, val)`, `update()` et `execute(f)`.
> On veut remettre l'image dans son orientation d'origine, de deux manières.

⚠️ **Pour que ce notebook fonctionne partout** (y compris sans pygame ni le fichier image), on **simule** le module avec une petite image de 8 × 8 « pixels » affichée en texte. Les fonctions de résolution sont **exactement celles à utiliser avec le vrai module** : la fin du notebook explique comment.

**Convention :** comme dans pygame, `(i, j)` = (colonne, ligne). `i` augmente vers la **droite**, `j` vers le **bas**, et `(0, 0)` est le coin **en haut à gauche**.

In [1]:
# ---------- Simulation du module turnbuffon ----------
TAILLE = 8

ORIGINALE = [
    "########",
    "#.......",
    "#.......",
    "######..",
    "#.......",
    "#.......",
    "#.......",
    "#......o",
]

IMAGE = []        # IMAGE[j][i] = pixel de la colonne i, ligne j


def get_pixel(i, j):
    return IMAGE[j][i]


def set_pixel(i, j, val):
    IMAGE[j][i] = val


def update():
    pass          # avec pygame : rafraîchit la fenêtre


def execute(f):
    f()


def montre(titre):
    print(titre)
    for ligne in IMAGE:
        print("  " + " ".join(ligne))
    print()


def remet_image_tournee():
    # l'image de départ : l'originale tournée d'un quart de tour dans le sens inverse des aiguilles d'une montre
    global IMAGE
    IMAGE = [[ORIGINALE[i][TAILLE - 1 - j] for i in range(TAILLE)] for j in range(TAILLE)]


IMAGE_ORIGINALE = [list(ligne) for ligne in ORIGINALE]
IMAGE = IMAGE_ORIGINALE
montre("Image d'origine (ce qu'on veut retrouver) :")
remet_image_tournee()
montre("Image tournée (ce qu'on a au départ) :")

Image d'origine (ce qu'on veut retrouver) :
  # # # # # # # #
  # . . . . . . .
  # . . . . . . .
  # # # # # # . .
  # . . . . . . .
  # . . . . . . .
  # . . . . . . .
  # . . . . . . o

Image tournée (ce qu'on a au départ) :
  # . . . . . . o
  # . . . . . . .
  # . . # . . . .
  # . . # . . . .
  # . . # . . . .
  # . . # . . . .
  # . . # . . . .
  # # # # # # # #



## L'exemple d'utilisation de l'énoncé

```python
def rotation():
    for i in range(512):
        p = get_pixel(i, 0)
        for j in range(200, 300):
            set_pixel(i, j, p)
        update()
```

💡 Il ne tourne rien du tout ! Il sert seulement à montrer l'utilisation des fonctions : pour chaque colonne `i`, il lit le pixel de la **première ligne** et le recopie sur les lignes 200 à 299. On voit donc apparaître une bande horizontale. Le `update()` dans la boucle `for i` rafraîchit l'affichage après chaque colonne : on voit l'image se modifier progressivement.

---
## Question 1 : rotation avec un tableau

### (a) `rotation_tab`

**Idée :**
1. on **copie** toute l'image dans un tableau `tab` ;
2. on réécrit chaque pixel **à sa nouvelle place** en lisant dans la copie.

La copie est **indispensable** : si on écrivait directement dans l'image, on écraserait des pixels **avant de les avoir lus**.

**Où va le pixel `(i, j)` lors d'un quart de tour dans le sens des aiguilles d'une montre ?** Avec $N$ = taille de l'image :
- le coin haut-gauche `(0, 0)` va en haut à droite, en `(N-1, 0)` ;
- la première ligne (`j = 0`) devient la dernière colonne (`i = N-1`) ;
- la première colonne (`i = 0`) devient la première ligne (`j = 0`).

D'où la formule : **`(i, j)` → `(N - 1 - j, i)`**.

💡 Pour vérifier une formule de rotation, il suffit de tester **les 4 coins**.

In [2]:
def rotation_tab():
    N = TAILLE                      # 512 pour la vraie image
    tab = [[get_pixel(i, j) for j in range(N)] for i in range(N)]    # copie : tab[i][j]
    for i in range(N):
        for j in range(N):
            set_pixel(N - 1 - j, i, tab[i][j])
    update()


remet_image_tournee()
execute(rotation_tab)
montre("Après rotation_tab :")
assert IMAGE == IMAGE_ORIGINALE

Après rotation_tab :
  # # # # # # # #
  # . . . . . . .
  # . . . . . . .
  # # # # # # . .
  # . . . . . . .
  # . . . . . . .
  # . . . . . . .
  # . . . . . . o



⚠️ Si avec la vraie image le résultat est **à l'envers**, c'est qu'elle avait été tournée dans l'autre sens. Il suffit alors d'appliquer la rotation **3 fois**, ou d'utiliser la formule inverse : `(i, j)` → `(j, N - 1 - i)`.

### (b) Coût en mémoire

Le tableau contient **tous** les pixels : $512 \times 512 = 262\,144$ pixels.
Chaque pixel = 4 nombres entre 0 et 255 (rouge, vert, bleu, transparence), et chaque nombre tient sur **1 octet** (8 bits : $2^8 = 256$ valeurs).

$$262\,144 \times 4 = 1\,048\,576 \text{ octets} = 2^{20} \text{ octets} \approx \mathbf{1\ Mo}$$

💡 **En réalité, en Python, c'est bien plus :** chaque pixel est un tuple de 4 entiers, et un tuple occupe plusieurs dizaines d'octets en mémoire.

In [3]:
import sys
pixel = (12, 200, 37, 255)
taille_tuple = sys.getsizeof(pixel)
print("Un tuple de 4 entiers occupe", taille_tuple, "octets en Python")
print(f"Estimation pour 512 x 512 pixels : {512 * 512 * (taille_tuple + 8) / 2**20:.0f} Mo environ")

Un tuple de 4 entiers occupe 72 octets en Python
Estimation pour 512 x 512 pixels : 20 Mo environ


---
## Question 2 : rotation « diviser pour régner », sans tableau

**Principe :**
1. on découpe le carré en **4 carrés** deux fois plus petits ;
2. on tourne **récursivement** chacun des 4 carrés **sur place** ;
3. on **permute** les 4 carrés en suivant le schéma : haut-gauche → haut-droite → bas-droite → bas-gauche → haut-gauche.

```
  (i, j) ───────────► (i+t, j)
     ▲                    │
     │                    ▼
  (i, j+t) ◄──────── (i+t, j+t)
```

💡 **Pourquoi ça marche ?** Tourner toute l'image d'un quart de tour revient à :
- tourner **chaque quart** d'un quart de tour (le contenu de chaque carré est orienté correctement) ;
- puis **déplacer** les quarts : celui en haut à gauche part en haut à droite, etc.

**La permutation :** pour chaque position `(i, j)` du carré haut-gauche, on fait **tourner 4 pixels** à la fois. On les lit d'abord tous les 4 dans des variables, puis on les réécrit chacun à sa nouvelle place. On n'a besoin que de **4 variables**, et non d'un tableau de toute l'image : le coût en mémoire est négligeable.

**Paramètres :** `rotation(x, y, t)` tourne le carré de côté `t` dont le coin a pour coordonnées `(x, y)`. Dans le code, `m = t // 2` est le côté des 4 sous-carrés (c'est le `t` du schéma de l'énoncé).

⚠️ L'énoncé dit « coin **supérieur droit** » : avec `x = 0, y = 0` et le schéma, il s'agit en fait du coin **supérieur gauche** (le pixel `(0, 0)` est en haut à gauche). C'est très probablement une coquille de l'énoncé.

**Cas de base :** un carré de côté 1 (un seul pixel) n'a pas besoin d'être tourné.

In [4]:
def rotation(x=0, y=0, t=TAILLE):          # t=512 avec la vraie image
    if t == 1:
        return
    m = t // 2
    # 1. on tourne récursivement les 4 sous-carrés
    rotation(x, y, m)                      # haut-gauche
    rotation(x + m, y, m)                  # haut-droite
    rotation(x, y + m, m)                  # bas-gauche
    rotation(x + m, y + m, m)              # bas-droite
    # 2. on permute les 4 sous-carrés, pixel par pixel
    for i in range(x, x + m):
        for j in range(y, y + m):
            hg = get_pixel(i, j)
            hd = get_pixel(i + m, j)
            bd = get_pixel(i + m, j + m)
            bg = get_pixel(i, j + m)
            set_pixel(i + m, j, hg)        # haut-gauche -> haut-droite
            set_pixel(i + m, j + m, hd)    # haut-droite -> bas-droite
            set_pixel(i, j + m, bd)        # bas-droite  -> bas-gauche
            set_pixel(i, j, bg)            # bas-gauche  -> haut-gauche
    update()


remet_image_tournee()
execute(rotation)
montre("Après rotation (diviser pour régner) :")
assert IMAGE == IMAGE_ORIGINALE

Après rotation (diviser pour régner) :
  # # # # # # # #
  # . . . . . . .
  # . . . . . . .
  # # # # # # . .
  # . . . . . . .
  # . . . . . . .
  # . . . . . . .
  # . . . . . . o



### Vérification sur une grande image aléatoire

On compare les deux méthodes sur une image 64 × 64 remplie de valeurs aléatoires.

In [5]:
from random import randint, seed
seed(3)
TAILLE = 64
DEPART = [[randint(0, 9) for _ in range(TAILLE)] for _ in range(TAILLE)]

IMAGE = [ligne[:] for ligne in DEPART]
rotation_tab()
resultat_tab = [ligne[:] for ligne in IMAGE]

IMAGE = [ligne[:] for ligne in DEPART]
rotation(0, 0, TAILLE)
assert IMAGE == resultat_tab
print("Les deux méthodes donnent la même image ✔")

TAILLE = 8

Les deux méthodes donnent la même image ✔


---
## Avec le vrai module `turnbuffon.py`

Placer `turnbuffon.py` et `turnbuffon.png` dans le même dossier, puis lancer **sur son ordinateur** (pygame ouvre une fenêtre, ce qui ne marche pas dans Binder ou Colab) :

```python
from turnbuffon import *

TAILLE = 512

def rotation(x=0, y=0, t=512):
    ...   # le code ci-dessus

execute(rotation)
```

💡 Avec `update()` à la fin de chaque appel, on **voit** l'algorithme travailler : d'abord de tout petits carrés tournent, puis des carrés de plus en plus grands.

### Bilan des deux méthodes

| | `rotation_tab` | `rotation` (diviser pour régner) |
|:---|:---|:---|
| Mémoire supplémentaire | toute l'image (≈ 1 Mo, bien plus en Python) | 4 variables (+ la pile des appels récursifs : $\log(512) = 9$ niveaux) |
| Nombre de lectures et d'écritures | $N^2$ | $N^2$ à chaque niveau, soit $N^2 \log(N)$ en tout |

Ici, « diviser pour régner » **économise la mémoire**, au prix d'un peu plus de calculs.